In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Verify data files are accessible
import os
data_dir = "/content/drive/MyDrive/MasterThesis/MasterThesis/test_data"
print("Files in test_data:")
for f in os.listdir(data_dir):
    print(f"  {f}")

Mounted at /content/drive
Files in test_data:
  dbpedia.ttl
  rules_150minutes.txt
  original_train.ttl
  original_train_short.ttl
  rules_150minutes_short_150rules.txt
  rules_150minutes_short_100rules.txt
  rules_150minutes_short_200rules.txt
  rules_150minutes_short_500rules.txt
  rules_300minutes.txt


In [2]:
%%bash
# Install OpenJDK 11 and SBT
apt-get update -qq
apt-get install -y -qq openjdk-11-jdk-headless > /dev/null 2>&1
echo "deb https://repo.scala-sbt.org/scalasbt/debian all main" | tee /etc/apt/sources.list.d/sbt.list
curl -sL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0x2EE0EA64E40A89B84B2DF73499E82A75642AC823" | apt-key add
apt-get update -qq
apt-get install -y -qq sbt > /dev/null 2>&1
java -version
sbt --version

deb https://repo.scala-sbt.org/scalasbt/debian all main
OK
sbt runner version: 1.12.8


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: https://repo.scala-sbt.org/scalasbt/debian/dists/all/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)

[info] sbt runner (sbt-the-shell-script) is a runner to run any declared version of sbt.
[info] Actual version of the sbt is declared using project/build.properties for each build.


In [3]:
%%bash
# Clone RDFRules library (shallow clone to save time)
cd /content
if [ ! -d "rdfrules" ]; then
    git clone --depth 1 https://github.com/propi/rdfrules.git
fi
ls -la rdfrules/

total 240
drwxr-xr-x 13 root root  4096 Apr  6 20:33 .
drwxr-xr-x  1 root root  4096 Apr  6 20:33 ..
-rw-r--r--  1 root root  1757 Apr  6 20:33 build.sbt
drwxr-xr-x  4 root root  4096 Apr  6 20:33 core
drwxr-xr-x  6 root root  4096 Apr  6 20:33 experiments
drwxr-xr-x  4 root root  4096 Apr  6 20:33 experiments_amie2
drwxr-xr-x  4 root root  4096 Apr  6 20:33 experiments_amie3
drwxr-xr-x  4 root root  4096 Apr  6 20:33 experiments_kgc
drwxr-xr-x  8 root root  4096 Apr  6 20:33 .git
-rw-r--r--  1 root root   378 Apr  6 20:33 .gitattributes
drwxr-xr-x  3 root root  4096 Apr  6 20:33 .github
-rw-r--r--  1 root root  1253 Apr  6 20:33 .gitignore
drwxr-xr-x  5 root root  4096 Apr  6 20:33 gui
drwxr-xr-x  5 root root  4096 Apr  6 20:33 http
-rw-r--r--  1 root root 35147 Apr  6 20:33 LICENSE
-rw-r--r--  1 root root 40681 Apr  6 20:33 measures.png
-rw-r--r--  1 root root   721 Apr  6 20:33 pack-release.sh
drwxr-xr-x  2 root root  4096 Apr  6 20:33 project
-rw-r--r--  1 root root 25568 Apr  6 20

Cloning into 'rdfrules'...
Updating files: 100% (557/557), done.


In [4]:
%%bash
mkdir -p /content/ScalaMeasuring/project
mkdir -p /content/ScalaMeasuring/src/main/scala/com/masterthesis

cat > /content/ScalaMeasuring/build.sbt << 'BUILDSBT'
name := "scala-measuring"
organization := "com.masterthesis"
version := "1.0.0"
scalaVersion := "2.13.8"
scalacOptions := Seq("-unchecked", "-deprecation", "-feature", "-encoding", "utf8")

lazy val root = project
  .in(file("."))
  .dependsOn(rdfrules_core)

lazy val rdfrules_core = ProjectRef(file("../rdfrules"), "core")

libraryDependencies += "org.slf4j" % "slf4j-simple" % "1.7.36"

fork := true
javaOptions += "-Xmx10g"
BUILDSBT

cat > /content/ScalaMeasuring/project/build.properties << 'PROPS'
sbt.version=1.3.7
PROPS

cat > /content/ScalaMeasuring/project/plugins.sbt << 'PLUGINS'
logLevel := Level.Warn
addSbtPlugin("org.scala-js" % "sbt-scalajs" % "1.10.0")
addSbtPlugin("org.xerial.sbt" % "sbt-pack" % "0.14")
PLUGINS

echo 'Project skeleton created.'

Project skeleton created.


In [5]:
%%writefile /content/ScalaMeasuring/src/main/scala/com/masterthesis/SupportBenchmark.scala
package com.masterthesis

import com.github.propi.rdfrules.data.Graph
import com.github.propi.rdfrules.index.Index
import com.github.propi.rdfrules.rule.{Measure, ResolvedAtom, ResolvedRule}
import com.github.propi.rdfrules.ruleset.Ruleset
import com.github.propi.rdfrules.utils.{Debugger, ForEach}

import scala.io.Source

object SupportBenchmark {

  private var prefixMap: Map[String, String] = Map.empty

  private def parsePrefixesFromTtl(ttlPath: String): Map[String, String] = {
    val src = scala.io.Source.fromFile(ttlPath, "UTF-8")
    try {
      src.getLines()
        .map(_.trim)
        .filter(l => l.toLowerCase.startsWith("@prefix") || l.toLowerCase.startsWith("prefix"))
        .flatMap { line =>
          val colonIdx = line.indexOf(':')
          if (colonIdx < 0) None
          else {
            val nameStart = if (line.toLowerCase.startsWith("@prefix")) 7 else 6
            val name = line.substring(nameStart, colonIdx).trim
            val uriStart = line.indexOf('<', colonIdx)
            val uriEnd   = if (uriStart >= 0) line.indexOf('>', uriStart + 1) else -1
            if (uriStart < 0 || uriEnd < 0) None
            else Some(s"$name:" -> line.substring(uriStart + 1, uriEnd))
          }
        }
        .toMap
    } finally src.close()
  }

  private def expandUri(raw: String): String = {
    if (raw.startsWith("<") && raw.endsWith(">")) {
      raw
    } else {
      prefixMap.find { case (prefix, _) => raw.startsWith(prefix) } match {
        case Some((prefix, ns)) => s"<$ns${raw.substring(prefix.length)}>"
        case None => s"<$raw>"
      }
    }
  }

  private def parseAtom(s: String): ResolvedAtom = {
    val inner = s.trim.stripPrefix("(").stripSuffix(")").trim
    val parts = inner.split("\\s+")
    if (parts.length != 3)
      throw new IllegalArgumentException(s"Cannot parse atom (expected 3 parts): '$s'")
    ResolvedAtom.parse(parts(0), expandUri(parts(1)), parts(2))
  }

  private def parseRuleLine(line: String): ResolvedRule = {
    val pipeIdx = line.lastIndexOf('|')
    val rulePart = if (pipeIdx >= 0) line.substring(0, pipeIdx).trim else line.trim
    val measuresPart = if (pipeIdx >= 0) line.substring(pipeIdx + 1).trim else ""

    val arrowIdx = rulePart.indexOf("=>")
    if (arrowIdx < 0)
      throw new IllegalArgumentException(s"No '=>' found in rule: '$line'")

    val bodyStr = rulePart.substring(0, arrowIdx).trim
    val headStr = rulePart.substring(arrowIdx + 2).trim
    val head = parseAtom(headStr)

    val bodyAtoms: IndexedSeq[ResolvedAtom] = if (bodyStr.isEmpty) {
      IndexedSeq.empty
    } else {
      val atomPattern = """\(([^)]+)\)""".r
      atomPattern.findAllMatchIn(bodyStr).map(m => parseAtom(m.group(0))).toIndexedSeq
    }

    val measures = scala.collection.mutable.ArrayBuffer.empty[Measure]
    if (measuresPart.nonEmpty) {
      for (p <- measuresPart.split(",").map(_.trim)) {
        val kv = p.split(":\\s*", 2)
        if (kv.length == 2) kv(0).trim match {
          case "Support"      => measures += Measure.Support(kv(1).trim.toInt)
          case "HeadCoverage" => measures += Measure.HeadCoverage(kv(1).trim.toDouble)
          case "HeadSupport"  => measures += Measure.HeadSupport(kv(1).trim.toInt)
          case "HeadSize"     => measures += Measure.HeadSize(kv(1).trim.toInt)
          case _              =>
        }
      }
    }
    ResolvedRule(bodyAtoms, head, measures.toSeq: _*)
  }

  def main(args: Array[String]): Unit = {
    if (args.length < 2) {
      System.err.println("Usage: SupportBenchmark <train.ttl> <rules.txt>")
      System.exit(1)
    }

    val ttlPath   = args(0)
    val rulesPath = args(1)

    prefixMap = parsePrefixesFromTtl(ttlPath)
    println(s"Loaded ${prefixMap.size} prefix(es) from $ttlPath")

    implicit val debugger: Debugger = Debugger.EmptyDebugger

    println(s"Loading TTL: $ttlPath")
    val t0 = System.nanoTime()
    val index: Index = Graph(ttlPath).toDataset.index
    val t1 = System.nanoTime()
    println(f"Index built in ${(t1 - t0) / 1e9}%.3f s")

    println(s"Parsing rules: $rulesPath")
    val src = Source.fromFile(rulesPath, "UTF-8")
    val ruleLines = try { src.getLines().filter(_.trim.nonEmpty).toVector } finally { src.close() }
    val resolvedRules: IndexedSeq[ResolvedRule] = ruleLines.map(parseRuleLine)
    println(s"Parsed ${resolvedRules.size} rules")

    print("Warming up index (1 throwaway rule) ... ")
    val warmupT0 = System.nanoTime()
    Ruleset(index, ForEach.from(resolvedRules.take(1))).setParallelism(1)
      .computeSupport(minSupport = 1, injectiveMapping = true).resolvedRules.foreach(_ => ())
    val warmupT1 = System.nanoTime()
    println(f"done in ${(warmupT1 - warmupT0) / 1e9}%.3f s")

    val ruleset: Ruleset = Ruleset(index, ForEach.from(resolvedRules)).setParallelism(1)

    println("Computing support (single-threaded to save memory) ...")
    println("\n========== RESULTS (streaming) ==========")
    var count = 0
    val perRuleTimes    = scala.collection.mutable.ArrayBuffer.empty[Double]
    val perRuleBodySizes = scala.collection.mutable.ArrayBuffer.empty[Int]
    val t2 = System.nanoTime()
    var lastRuleTime = t2
    ruleset.computeSupport(minSupport = 1, injectiveMapping = true).resolvedRules.foreach { r =>
      val now = System.nanoTime()
      val ruleMs = (now - lastRuleTime) / 1e6
      lastRuleTime = now
      count += 1
      perRuleTimes += ruleMs
      perRuleBodySizes += r.body.size
      val supp = r.measures.get(Measure.Support).map(_.value).getOrElse(-1)
      val hc   = r.measures.get(Measure.HeadCoverage).map(_.value).getOrElse(0.0)
      val hs   = r.measures.get(Measure.HeadSupport).map(_.value).getOrElse(-1)
      val hsz  = r.measures.get(Measure.HeadSize).map(_.value).getOrElse(-1)
      println(f"[$count%4d] ${ruleMs}%10.1f ms | Support: $supp, HC: $hc, HS: $hs, HSz: $hsz | $r")
      System.out.flush()
    }
    val t3 = System.nanoTime()
    val supportTimeSec = (t3 - t2) / 1e9

    val sortedTimes = perRuleTimes.sorted
    val avgMs = if (perRuleTimes.nonEmpty) perRuleTimes.sum / perRuleTimes.size else 0.0
    val medMs = if (sortedTimes.nonEmpty) sortedTimes(sortedTimes.size / 2) else 0.0
    val minMs = if (sortedTimes.nonEmpty) sortedTimes.head else 0.0
    val maxMs = if (sortedTimes.nonEmpty) sortedTimes.last else 0.0
    val p95Ms = if (sortedTimes.nonEmpty) sortedTimes((sortedTimes.size * 0.95).toInt.min(sortedTimes.size - 1)) else 0.0

    println("\n========== TIMING SUMMARY ==========")
    println(f"Index build time:          ${(t1 - t0) / 1e9}%.3f s")
    println(f"Warmup time:               ${(warmupT1 - warmupT0) / 1e9}%.3f s")
    println(f"Support counting time:     ${supportTimeSec}%.3f s")
    println(f"Total rules parsed:        ${resolvedRules.size}")
    println(f"Rules surviving (supp>=1): $count")
    println(f"Avg time per rule:         ${avgMs}%.1f ms")
    println(f"Median time per rule:      ${medMs}%.1f ms")
    println(f"Min time per rule:         ${minMs}%.1f ms")
    println(f"Max time per rule:         ${maxMs}%.1f ms")
    println(f"P95 time per rule:         ${p95Ms}%.1f ms")
    println(f"Total wall time:           ${(t3 - t0) / 1e9}%.3f s")

    println("\n========== STATISTICS BY BODY SIZE ==========")
    val grouped = perRuleTimes.zip(perRuleBodySizes).groupBy(_._2)
    for (bodySize <- grouped.keys.toSeq.sorted) {
      val times = grouped(bodySize).map(_._1).sorted
      val n = times.size
      val avg = times.sum / n
      val med = times(n / 2)
      val p95 = times(((n * 0.95).toInt).min(n - 1))
      println(f"Body size $bodySize ($n rules): avg=$avg%.1f ms, median=$med%.1f ms, min=${times.head}%.1f ms, max=${times.last}%.1f ms, P95=$p95%.1f ms")
    }
  }
}


Writing /content/ScalaMeasuring/src/main/scala/com/masterthesis/SupportBenchmark.scala


In [6]:
%%bash
DATA_DIR="/content/drive/MyDrive/MasterThesis/MasterThesis/test_data"
echo "Data files:"
ls -la "$DATA_DIR/"

Data files:
total 436680
-rw------- 1 root root  26674438 Jan  8 22:35 dbpedia.ttl
-rw------- 1 root root  60544080 Mar 26 05:43 original_train_short.ttl
-rw------- 1 root root 359382237 Mar 22 01:31 original_train.ttl
-rw------- 1 root root     18414 Mar 27 04:16 rules_150minutes_short_100rules.txt
-rw------- 1 root root     29959 Mar 27 04:14 rules_150minutes_short_150rules.txt
-rw------- 1 root root     41061 Mar 28 03:18 rules_150minutes_short_200rules.txt
-rw------- 1 root root    106567 Mar 28 03:46 rules_150minutes_short_500rules.txt
-rw------- 1 root root    158732 Mar 22 01:15 rules_150minutes.txt
-rw------- 1 root root    201966 Mar 28 10:04 rules_300minutes.txt


In [7]:
%%bash
cd /content/ScalaMeasuring

# Compile the project (first run downloads all dependencies)
sbt compile 2>&1

# Export the full runtime classpath to a file so we can run via java directly
echo "--- Exporting classpath ---"
sbt --no-colors "export runtime:fullClasspath" 2>&1 | tail -1 > /content/ScalaMeasuring/classpath.txt
echo "Classpath saved. SBT can now be killed to free memory."

[info] [launcher] getting org.scala-sbt sbt 1.3.7  (this may take some time)...
[info] [launcher] getting Scala 2.12.10 (for sbt)...
[info] Loading settings for project scalameasuring-build from plugins.sbt ...
[info] Loading project definition from /content/ScalaMeasuring/project
[warn] There may be incompatibilities among your library dependencies; run 'evicted' to see detailed eviction warnings.
[info] Loading settings for project root from build.sbt ...
[info] Loading settings for project rdfrules-build from plugins.sbt ...
[info] Loading project definition from /content/rdfrules/project
[warn] There may be incompatibilities among your library dependencies; run 'evicted' to see detailed eviction warnings.
[info] Loading settings for project root from build.sbt ...
[info] Loading settings for project gui from build.sbt ...
[info] Loading settings for project http from build.sbt ...
[info] Loading settings for project core from build.sbt ...
[info] Set current project to scala-measur

In [8]:
%%bash
# Kill any lingering SBT/Java processes to free ~2GB of RAM
pkill -f sbt.boot 2>/dev/null || true
sleep 2

# Read the exported classpath
CP=$(cat /content/ScalaMeasuring/classpath.txt)
DATA_DIR="/content/drive/MyDrive/MasterThesis/MasterThesis/test_data"

# Run benchmark directly with java (not sbt run) to maximize available RAM
# -Xmx11g: nearly all of Colab's 12GB
# -XX:+UseSerialGC: no GC thread overhead for single-threaded workload
java -Xmx11g -XX:+UseSerialGC -cp "$CP" com.masterthesis.SupportBenchmark \
  "$DATA_DIR/original_train.ttl" \
  "$DATA_DIR/rules_150minutes_short_500rules.txt"

Loaded 7 prefix(es) from /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/original_train.ttl
Loading TTL: /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/original_train.ttl
Index built in 0.102 s
Parsing rules: /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/rules_150minutes_short_500rules.txt
Parsed 500 rules
Warming up index (1 throwaway rule) ... done in 232.485 s
Computing support (single-threaded to save memory) ...

========== RESULTS (streaming) ==========
[   1]       37.4 ms | Support: 1062, HC: 1.0, HS: 1062, HSz: 1062 | (?a biolink:related_to ?b) -> (?a biolink:has_chemical_role ?b) | support: 1062, headCoverage: 1.0, headSize: 1062, headSupport: 1062
[   2]     7098.0 ms | Support: 5082, HC: 0.06363476997821242, HS: 79862, HSz: 79862 | (?a biolink:has_input ?b) -> (?a biolink:has_output ?b) | support: 5082, headCoverage: 0.06363476997821242, headSize: 79862, headSupport: 79862
[   3]      142.3 ms | Support: 5082, HC: 0.06363476997821242, HS:

[main] INFO com.github.propi.rdfrules.utils.Debugger - Predicates trimming.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Subjects indexing.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Subjects trimming.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Objects indexing.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Objects trimming.


In [9]:
%%bash
# Kill any lingering SBT/Java processes to free ~2GB of RAM
pkill -f sbt.boot 2>/dev/null || true
sleep 2

# Read the exported classpath
CP=$(cat /content/ScalaMeasuring/classpath.txt)
DATA_DIR="/content/drive/MyDrive/MasterThesis/MasterThesis/test_data"

# Run benchmark directly with java (not sbt run) to maximize available RAM
# -Xmx11g: nearly all of Colab's 12GB
# -XX:+UseSerialGC: no GC thread overhead for single-threaded workload
java -Xmx11g -XX:+UseSerialGC -cp "$CP" com.masterthesis.SupportBenchmark \
  "$DATA_DIR/original_train.ttl" \
  "$DATA_DIR/rules_150minutes.txt"

Loaded 7 prefix(es) from /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/original_train.ttl
Loading TTL: /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/original_train.ttl
Index built in 0.168 s
Parsing rules: /content/drive/MyDrive/MasterThesis/MasterThesis/test_data/rules_150minutes.txt
Parsed 736 rules
Warming up index (1 throwaway rule) ... done in 223.885 s
Computing support (single-threaded to save memory) ...

========== RESULTS (streaming) ==========
[   1]       48.4 ms | Support: 1062, HC: 1.0, HS: 1062, HSz: 1062 | (?a biolink:related_to ?b) -> (?a biolink:has_chemical_role ?b) | support: 1062, headCoverage: 1.0, headSize: 1062, headSupport: 1062
[   2]      316.0 ms | Support: 5082, HC: 0.06363476997821242, HS: 79862, HSz: 79862 | (?a biolink:has_input ?b) -> (?a biolink:has_output ?b) | support: 5082, headCoverage: 0.06363476997821242, headSize: 79862, headSupport: 79862
[   3]       76.1 ms | Support: 5082, HC: 0.06363476997821242, HS: 79862, HSz: 79

[main] INFO com.github.propi.rdfrules.utils.Debugger - Predicates trimming.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Subjects indexing.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Subjects trimming.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Objects indexing.
[main] INFO com.github.propi.rdfrules.utils.Debugger - Objects trimming.


In [ ]:
%%bash
# Kill any lingering SBT/Java processes to free ~2GB of RAM
pkill -f sbt.boot 2>/dev/null || true
sleep 2

# Read the exported classpath
CP=$(cat /content/ScalaMeasuring/classpath.txt)
DATA_DIR="/content/drive/MyDrive/MasterThesis/MasterThesis/test_data"

# Run benchmark directly with java (not sbt run) to maximize available RAM
# -Xmx11g: nearly all of Colab's 12GB
# -XX:+UseSerialGC: no GC thread overhead for single-threaded workload
java -Xmx11g -XX:+UseSerialGC -cp "$CP" com.masterthesis.SupportBenchmark \
  "$DATA_DIR/original_train.ttl" \
  "$DATA_DIR/rules_300minutes.txt"